In [1]:
pip install llama-cpp-python huggingface_hub transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 134.6 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 317.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 174.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 196.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 386.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp311-cp311-linux_x86_64.whl size=4500177 sha256=236d0af0ac32ab09766e0bc900008f38e430d4f2998c004d2018e73cf2a1f59f
  Stored in directory: /root/.cache/pip/wheels/d8/5b/e5/a7d4b5765da347d314e8155197440c9995a962f8e4a5f52b23
Successfully built llama-cpp-python

[noti

In [6]:
%env CMAKE_ARGS=-DGGML_CUDA=on
%pip install llama-cpp-python[server] --force-reinstall --no-cache-dir

env: CMAKE_ARGS=-DGGML_CUDA=on
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 261.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 250.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 701.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 783.6 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp311-cp311-linux_x86_64.whl size=49158903 sha256=eadf78db5a0823d7e913a6f756b2fead028d5f9b1cbd97a8ca737bdf374d9c94
  Stored in directory: /tmp/pip-ephem-wheel-cache-tmni5cpy/wheels/d8/5b/e5/a7d4b5765da347d314e8155197440c9995a962f8e4a5f52b23
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.9.0
    Uninstalling typing_

In [11]:
!pip install cmake

  Using cached cmake-4.2.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.5 kB)
Using cached cmake-4.2.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (28.9 MB)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
import os
import sys
import shutil
import subprocess
from huggingface_hub import HfApi, login, snapshot_download

HF_TOKEN = os.getenv("HF_TOKEN")

# 변환할 원본 모델
SOURCE_MODEL_ID = "WindyAle/kanana-nano-2.1B-customer-emotional"

# 업로드할 리포지토리
TARGET_REPO_ID = "WindyAle/kanana-nano-2.1B-customer-emotional-gguf"

# 파일명 설정
GGUF_FILENAME_FP16 = "kanana-nano-2.1B-customer-emotional.gguf"
GGUF_FILENAME_Q4 = "kanana-nano-2.1B-customer-emotional-Q4_K_M.gguf"

BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "downloaded_model")


LLAMA_CPP_DIR = os.path.join(BASE_DIR, "llama.cpp")

In [8]:
def run_command(command, cwd=None):
    print(f"실행: {' '.join(command)}")
    try:
        subprocess.run(command, check=True, cwd=cwd)
    except subprocess.CalledProcessError as e:
        print(f"오류 발생: {e}")
        sys.exit(1)

In [9]:
def load_model():
    if not HF_TOKEN:
        print("오류: HF_TOKEN 없음")
        sys.exit(1)
    login(token=HF_TOKEN)
    
    print(f"원본 모델: {SOURCE_MODEL_ID}")
    print(f"타겟 리포지토리: {TARGET_REPO_ID}")

    print(f"\n모델 다운로드: ({SOURCE_MODEL_ID})")
    if os.path.exists(MODEL_DIR):
        shutil.rmtree(MODEL_DIR)
        
    try:
        snapshot_download(
            repo_id=SOURCE_MODEL_ID,
            local_dir=MODEL_DIR,
            local_dir_use_symlinks=False,
            ignore_patterns=["*.gguf", "*.bin", "*.pth"]
        )
        print("✅ 다운로드 완료")
    except Exception as e:
        print(f"다운로드 실패: {e}")
        sys.exit(1)

In [10]:
def main():
    if not os.path.exists(LLAMA_CPP_DIR):
        run_command(["git", "clone", "https://github.com/ggerganov/llama.cpp.git"])
    
    # 필수 패키지 설치
    run_command([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=LLAMA_CPP_DIR)
    
    # CMake 빌드 (양자화 도구 생성)
    print("llama.cpp 빌드 중...")
    build_dir = os.path.join(LLAMA_CPP_DIR, "build")
    run_command(["cmake", "-B", build_dir, "-DGGML_NATIVE=OFF"], cwd=LLAMA_CPP_DIR)
    run_command(["cmake", "--build", build_dir, "--config", "Release", "-j", "4"], cwd=LLAMA_CPP_DIR)
    
    # 양자화 실행 파일 위치 찾기
    quantize_bin = os.path.join(build_dir, "bin", "llama-quantize")
    if not os.path.exists(quantize_bin):
        # 경로가 다를 경우 (루트 빌드 등) 대비
        quantize_bin = os.path.join(build_dir, "llama-quantize")
        
    if not os.path.exists(quantize_bin):
        print("❌ 오류: llama-quantize 바이너리 빌드 실패.")
        sys.exit(1)

    print(f"\n[3/5] GGUF 변환 (FP16) 수행 중...")
    convert_script = os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")
    
    run_command([
        sys.executable, convert_script,
        MODEL_DIR,
        "--outfile", GGUF_FILENAME_FP16,
        "--outtype", "f16"
    ])
    
    if not os.path.exists(GGUF_FILENAME_FP16):
        print("❌ 변환 실패:")
        sys.exit(1)

    print(f"\n4bit 양자화")
    
    run_command([
        quantize_bin,
        GGUF_FILENAME_FP16,
        GGUF_FILENAME_Q4,
        "Q4_K_M"
    ])
    
    if not os.path.exists(GGUF_FILENAME_Q4):
        print("❌ 양자화 실패")
        sys.exit(1)
        
    print(f"✅ 양자화 완료: {GGUF_FILENAME_Q4}")
    
    # 용량 확보를 위해 FP16 파일 삭제
    os.remove(GGUF_FILENAME_FP16)
    shutil.rmtree(MODEL_DIR) 



if __name__ == "__main__":
    main()

실행: git clone https://github.com/ggerganov/llama.cpp.git


Cloning into 'llama.cpp'...
Updating files: 100% (2213/2213), done.


실행: /usr/bin/python -m pip install -r requirements.txt
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 87.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 266.7 MB/s eta 0:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.4.1+cu124 requires torch==2.4.1, but you have torch 2.6.0+cpu which is incompatible.
torchvision 0.19.1+cu124 requires torch==2.4.1, but you have torch 2.6.0+cpu which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


llama.cpp 빌드 중...
실행: cmake -B /workspace/llama.cpp/build -DGGML_NATIVE=OFF


FileNotFoundError: [Errno 2] No such file or directory: 'cmake'

In [ ]:
# 업로드
print(f"\n[5/5] Hugging Face 업로드 시작 ({TARGET_REPO_ID})...")
login(token=HF_TOKEN)    
api = HfApi()

api.create_repo(repo_id=TARGET_REPO_ID, exist_ok=True, repo_type="model")

try:
    api.upload_file(
        path_or_fileobj=GGUF_FILENAME_Q4,
        path_in_repo=GGUF_FILENAME_Q4,
        repo_id=TARGET_REPO_ID,
        repo_type="model"
    )
    print("\n업로드 완료")
    print(f"모델 링크: https://huggingface.co/{TARGET_REPO_ID}")
    
except Exception as e:
    print(f"❌ 업로드 중 오류 발생: {e}")


[5/5] Hugging Face 업로드 시작 (WindyAle/Kanana-nano-2.1B-Finance-v1-GGUF)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 모든 작업이 완료되었습니다!
🔗 모델 링크: https://huggingface.co/WindyAle/Kanana-nano-2.1B-Finance-v1-GGUF
